# 02 · Đặc trưng âm thanh và baseline phân loại mood

**Câu hỏi của notebook:** chỉ *nghe* âm thanh (không nhìn tag thể loại), mô hình đoán đúng nhóm mood tới đâu, và có hơn mốc "chỉ dùng thể loại" không?

**Luồng làm việc**
1. Đọc `data/index_mood.csv` (do notebook 01 tạo): các bài sạch mood và có audio.
2. Mỗi bài: cắt 30 s giữa bài → mel-spectrogram → **vector đặc trưng** có số chiều cố định.
3. Lưu bảng đặc trưng ra file (không phải tính lại).
4. Chia train/test **theo artist**.
5. Chuẩn hóa, huấn luyện baseline, đo bằng macro-F1, so với 3 mốc.
6. Ablation: bỏ từng nhóm đặc trưng xem cái nào quan trọng.

**Trước khi chạy:** phải chạy xong notebook 01 (để có `data/index_mood.csv`) và đặt `AUDIO_DIR` đúng như notebook 01.

> Nguyên tắc cả dự án: thử nghiệm ở notebook trước; hàm nào ổn định thì chuyển sang `src/*.py`.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.fft
import librosa
import librosa.display
from joblib import Parallel, delayed, parallel_config
from IPython.display import display

# ============ CẤU HÌNH — chỉnh ở đây ============
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
AUDIO_DIR = Path(os.environ.get("JAMENDO_AUDIO_DIR", ROOT / "data" / "audio-low")).expanduser()
INDEX_CSV = ROOT / "data" / "index_mood.csv"        # tạo bởi notebook 01
FEATURE_CSV = ROOT / "data" / "features_mood.csv"   # bảng đặc trưng (cache)

SR = 22050          # tần số lấy mẫu (Hz), giống notebook 01
CLIP_SEC = 30       # đoạn lấy ở giữa bài (giây)
N_FFT, HOP, N_MELS = 2048, 512, 128
N_MFCC = 20         # số hệ số MFCC giữ lại
N_JOBS = -1         # -1 = dùng mọi nhân CPU
MAX_TRACKS = None   # đặt vd 40 để chạy thử nhanh; None = tất cả
SEED = 0            # cố định ngẫu nhiên để kết quả lặp lại được
# ================================================

plt.rcParams["figure.dpi"] = 110
print("ROOT =", ROOT)
print("AUDIO_DIR =", AUDIO_DIR)

---
## Phần 1 — Từ bài hát đến một vector số

### 1.1. Đọc danh sách bài
`index_mood.csv` có mỗi bài một dòng: `track_id`, `artist_id`, `path`, `duration`, `quadrant` (nhóm mood, là **nhãn**) và `genre_coarse` (thể loại lớn, chỉ để làm mốc so sánh, **không** đưa vào mô hình).

In [ ]:
def audio_path(rel_path, audio_dir=AUDIO_DIR):
    # 'PATH' trong TSV là '48/948.mp3'; file audio-low thật là '48/948.low.mp3'.
    return Path(audio_dir) / str(rel_path).replace(".mp3", ".low.mp3")


if not INDEX_CSV.exists():
    raise FileNotFoundError(f"Chưa có {INDEX_CSV}. Hãy chạy hết notebook 01 trước.")

index_df = pd.read_csv(INDEX_CSV, dtype={"track_id": str})
index_df["has_audio"] = [audio_path(p).exists() for p in index_df["path"]]
index_df = index_df[index_df["has_audio"]].reset_index(drop=True)
if MAX_TRACKS:
    index_df = index_df.sample(MAX_TRACKS, random_state=SEED).reset_index(drop=True)

print(f"Số bài dùng: {len(index_df)} | số artist: {index_df['artist_id'].nunique()}")
print("\nSố bài mỗi nhóm mood:")
print(index_df["quadrant"].value_counts().to_string())
index_df.head(3)

### 1.2. Nạp đoạn 30 s ở giữa bài
Bài dài tới cả giờ nên không nạp hết. Ta lấy **30 s ở giữa** vì phần đầu/cuối thường là đoạn dạo/outro, không đại diện cho cả bài. Bài ngắn hơn 30 s thì nạp hết.

In [ ]:
def middle_clip(path, duration, sr=SR, clip=CLIP_SEC):
    offset = max(0.0, duration / 2 - clip / 2)
    y, _ = librosa.load(path, sr=sr, mono=True, offset=offset, duration=clip)
    return y


row = index_df.iloc[0]
y = middle_clip(audio_path(row["path"]), row["duration"])
print(f"Bài {row['track_id']} ({row['quadrant']}): {len(y)} mẫu = {len(y)/SR:.1f} s, trung bình biên độ |y| = {np.abs(y).mean():.3f}")

### 1.3. MFCC: log-mel rồi DCT

Từ notebook 01 ta có **mel-spectrogram**: mỗi khung thời gian là 128 con số (năng lượng trong 128 băng mel). Hai vấn đề:
- 128 số là nhiều, và các băng kề nhau **tương quan mạnh** (các bộ lọc mel chồng lấn nhau).
- Đưa thẳng vào mô hình thì dư thừa.

**MFCC** (Mel-Frequency Cepstral Coefficients) xử lý bằng hai bước:
1. Lấy **log** năng lượng mel (tai người nghe theo log, và log biến phép nhân thành phép cộng).
2. Áp **DCT** (biến đổi cosine rời rạc) dọc theo trục tần số:

$$c_k=\sum_{m=0}^{M-1}\log E_m\,\cos\!\Big[\frac{\pi k}{M}\big(m+\tfrac12\big)\Big],\qquad k=0,1,\dots$$

với $E_m$ là năng lượng băng mel thứ $m$, $M=128$. DCT **khử tương quan** và **dồn thông tin vào các hệ số đầu**:
- $c_0$: mức năng lượng chung (gần như độ to).
- $c_1$: nghiêng giữa tần số thấp và cao (bài "sáng" hay "tối").
- $c_k$ lớn dần: chi tiết ngày càng mịn của phổ (như các hài âm và cao độ).

Ta chỉ giữ 20 hệ số đầu: đó là **đường bao phổ**, xấp xỉ **âm sắc** của nhạc. Phần chi tiết mịn bị bỏ đi.

Ô dưới **tự kiểm** điều này: tự tính DCT từ log-mel rồi so với hàm `librosa.feature.mfcc`.

In [ ]:
S_pow = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)) ** 2          # công suất, (1025, số khung)
logmel = librosa.power_to_db(librosa.feature.melspectrogram(S=S_pow, sr=SR, n_mels=N_MELS))   # (128, số khung)

mfcc_manual = scipy.fft.dct(logmel, axis=0, type=2, norm="ortho")[:N_MFCC]   # DCT dọc trục tần số, giữ 20 hệ số đầu
mfcc_lib = librosa.feature.mfcc(S=logmel, n_mfcc=N_MFCC)

print("logmel:", logmel.shape, "-> mfcc:", mfcc_lib.shape)
print("Tự tính DCT khớp librosa?", np.allclose(mfcc_manual, mfcc_lib))
print("Độ lệch tối đa:", np.abs(mfcc_manual - mfcc_lib).max())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
librosa.display.specshow(logmel, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[0])
axes[0].set_title("log-mel (128 băng)")
librosa.display.specshow(mfcc_lib, sr=SR, hop_length=HOP, x_axis="time", ax=axes[1])
axes[1].set_title("MFCC (20 hệ số)"); axes[1].set_ylabel("hệ số k")
axes[2].plot(mfcc_lib.mean(axis=1), "o-")
axes[2].set_title("trung bình mỗi hệ số theo thời gian"); axes[2].set_xlabel("hệ số k")
plt.tight_layout(); plt.show()

**Cách đọc hình:** ở hình giữa, vài hàng đầu (k nhỏ) có màu đậm hơn hẳn, tức là hệ số đầu có **biên độ lớn hơn nhiều** các hệ số sau; ở hình phải, các hệ số từ khoảng k = 6 trở đi chỉ quanh 0. Thang số các đặc trưng chênh nhau như vậy nên khi đưa vào mô hình **bắt buộc chuẩn hóa** (làm ở phần sau), nếu không các chiều có số lớn sẽ lấn át.

### 1.4. Các nhóm đặc trưng khác và vì sao chọn chúng

Mood thể hiện qua **năng lượng** (arousal), **âm sắc**, **hòa âm** (vui/buồn gắn với trưởng/thứ), và **nhịp**. Mỗi nhóm đặc trưng sau đo một mặt:

| Nhóm | Đặc trưng | Đo cái gì |
|---|---|---|
| `mfcc` | 20 hệ số MFCC | âm sắc (đường bao phổ) |
| `chroma` | 12 lớp cao độ (C, C#, ... B) | hòa âm: nốt nào đang vang, trưởng hay thứ |
| `spectral` | centroid, rolloff, bandwidth, flatness, contrast | độ "sáng", độ rộng phổ, mức giống tiếng ồn, độ tương phản đỉnh/đáy |
| `energy` | RMS, zero-crossing rate | độ to, độ gắt/ồn |
| `rhythm` | tempo, cường độ onset | tốc độ, độ dồn dập của nhịp |

Với mỗi đặc trưng theo thời gian (một giá trị mỗi khung) ta **tóm tắt bằng trung bình và độ lệch chuẩn** để ra số chiều cố định cho mọi bài: trung bình cho biết mức chung, độ lệch chuẩn cho biết **biến động** (bài nhiều thay đổi hay đều đều).

In [ ]:
def summarize(name, M, out):
    # M: (số đặc trưng, số khung). Mỗi hàng -> trung bình và độ lệch chuẩn theo thời gian.
    M = np.atleast_2d(M)
    for i, (mu, sd) in enumerate(zip(M.mean(axis=1), M.std(axis=1))):
        out[f"{name}_mean_{i:02d}"] = float(mu)
        out[f"{name}_std_{i:02d}"] = float(sd)


def extract_features(y, sr=SR):
    # Một đoạn audio -> dict {tên đặc trưng: giá trị}. Số chiều cố định (93).
    warnings.filterwarnings("ignore", message="Trying to estimate tuning")   # khung im lặng, vô hại
    mag = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP))            # biên độ
    S_pow = mag ** 2                                                       # công suất
    logmel = librosa.power_to_db(librosa.feature.melspectrogram(S=S_pow, sr=sr, n_mels=N_MELS))
    f = {}
    summarize("mfcc", librosa.feature.mfcc(S=logmel, n_mfcc=N_MFCC), f)                    # 40
    summarize("chroma", librosa.feature.chroma_stft(S=S_pow, sr=sr), f)                    # 24
    summarize("centroid", librosa.feature.spectral_centroid(S=mag, sr=sr), f)              # 2
    summarize("rolloff", librosa.feature.spectral_rolloff(S=mag, sr=sr), f)                # 2
    summarize("bandwidth", librosa.feature.spectral_bandwidth(S=mag, sr=sr), f)            # 2
    summarize("flatness", librosa.feature.spectral_flatness(S=mag), f)                     # 2
    summarize("contrast", librosa.feature.spectral_contrast(S=mag, sr=sr), f)              # 14
    summarize("zcr", librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP), f)   # 2
    summarize("rms", librosa.feature.rms(S=mag, frame_length=N_FFT), f)                    # 2
    onset = librosa.onset.onset_strength(S=logmel, sr=sr, hop_length=HOP)
    tempo, _ = librosa.beat.beat_track(onset_envelope=onset, sr=sr, hop_length=HOP)
    f["tempo"] = float(np.atleast_1d(tempo)[0])                                            # 1
    f["onset_mean"], f["onset_std"] = float(onset.mean()), float(onset.std())              # 2
    return f


FEATURE_GROUPS = {   # tiền tố tên đặc trưng -> nhóm (dùng cho ablation)
    "mfcc": "mfcc", "chroma": "chroma",
    "centroid": "spectral", "rolloff": "spectral", "bandwidth": "spectral",
    "flatness": "spectral", "contrast": "spectral",
    "zcr": "energy", "rms": "energy",
    "tempo": "rhythm", "onset": "rhythm",
}


def feature_group(col):
    return FEATURE_GROUPS[col.split("_")[0]]


t0 = time.time()
one = extract_features(y)
print(f"{len(one)} đặc trưng cho 1 bài, mất {time.time()-t0:.1f} s (lần đầu chậm hơn do biên dịch numba)")
print(pd.Series({g: sum(feature_group(c) == g for c in one) for g in ["mfcc", "chroma", "spectral", "energy", "rhythm"]}, name="số chiều"))

### 1.5. Trích đặc trưng cho cả tập (chạy song song, có lưu cache)

Chạy cho khoảng một nghìn bài mất vài phút đến vài chục phút tùy máy. Hai thiết kế để đỡ khổ:
- **Song song** theo bài bằng `joblib` (mỗi bài độc lập).
- **Cache**: kết quả lưu vào `features_mood.csv`. Lần chạy sau chỉ tính các bài chưa có, nên bị ngắt giữa chừng cũng không mất.

Bài nào lỗi (file hỏng, quá ngắn) thì bỏ qua và ghi lại, không làm sập cả vòng lặp.

In [ ]:
def _extract_one(track_id, rel_path, duration):
    try:
        y = middle_clip(audio_path(rel_path), duration)
        if len(y) < 5 * SR:
            return {"track_id": track_id, "error": f"quá ngắn ({len(y)/SR:.1f}s)"}
        f = extract_features(y)
        f["track_id"] = track_id
        return f
    except Exception as e:                      # file hỏng, thiếu codec...
        return {"track_id": track_id, "error": repr(e)[:120]}


cache = pd.read_csv(FEATURE_CSV, dtype={"track_id": str}) if FEATURE_CSV.exists() else pd.DataFrame()
done = set(cache["track_id"]) if len(cache) else set()
todo = index_df[~index_df["track_id"].isin(done)]
print(f"Đã có trong cache: {len(done)} | cần tính: {len(todo)}")

CHUNK = 200
t0 = time.time()
for start in range(0, len(todo), CHUNK):
    part = todo.iloc[start:start + CHUNK]
    with parallel_config(backend="loky", inner_max_num_threads=1):
        res = Parallel(n_jobs=N_JOBS)(
            delayed(_extract_one)(r.track_id, r.path, r.duration) for r in part.itertuples())
    ok = [r for r in res if "error" not in r]
    bad = [r for r in res if "error" in r]
    cache = pd.concat([cache, pd.DataFrame(ok)], ignore_index=True)
    cache.to_csv(FEATURE_CSV, index=False)
    print(f"  {start + len(part)}/{len(todo)} bài | lỗi: {len(bad)} {[b['error'] for b in bad][:2]} | {time.time()-t0:.0f}s")
print("Xong.")

### 1.6. Kiểm tra bảng đặc trưng
Trước khi dùng, luôn kiểm tra: có thiếu giá trị (NaN) không, số chiều đúng không, và **thang số** của các đặc trưng chênh nhau cỡ nào (để thấy vì sao phải chuẩn hóa).

In [ ]:
feat_cols = [c for c in cache.columns if c != "track_id"]
data = index_df.merge(cache, on="track_id", how="inner")
X_all = data[feat_cols]
print(f"Bảng đặc trưng: {len(data)} bài × {len(feat_cols)} đặc trưng")
print("Số ô NaN / vô cực:", int(X_all.isna().sum().sum()), "/", int(np.isinf(X_all.to_numpy()).sum()))
print("Số bài mỗi nhóm mood:"); print(data["quadrant"].value_counts().to_string())

rng = X_all.agg(["min", "max"]).T
rng["khoảng"] = rng["max"] - rng["min"]
print("\nThang số: 3 đặc trưng khoảng nhỏ nhất và lớn nhất")
display(pd.concat([rng.sort_values("khoảng").head(3), rng.sort_values("khoảng").tail(3)]).round(3))

---
## Phần 2 — Chia dữ liệu và baseline

### 2.1. Vì sao chia theo artist, không chia ngẫu nhiên theo bài

Cùng một nghệ sĩ thường có **cùng giọng hát, nhạc cụ, cách thu âm và phối khí**, và hay đặt tag mood giống nhau cho các bài của mình. Nếu bài của họ nằm ở **cả train lẫn test** thì mô hình có thể học "nghe giống artist X thì nhãn là Y" thay vì học mood. Điểm test khi đó **cao ảo** (rò rỉ dữ liệu).

Cách làm đúng: các bài của cùng một artist phải **nằm hết ở một phía**. Ta dùng `StratifiedGroupKFold`: chia **4 phần (fold)** sao cho
- mỗi artist chỉ nằm trong **một** fold (không rò rỉ artist),
- tỉ lệ 4 nhóm mood trong mỗi fold xấp xỉ nhau (giữ cân đối).

Mỗi fold lần lượt làm tập test, 3 fold còn lại làm train. Dữ liệu ít (127 bài ở nhóm nhỏ nhất) nên điểm rất **rung** theo cách chia. Vì vậy ta **lặp 5 lần với 5 cách chia khác nhau** (đổi `random_state`), tổng 20 fold, rồi lấy **trung bình ± độ lệch chuẩn** trên 20 fold đó.

Để **đo mức ảo**, ta cũng chạy cách chia ngẫu nhiên theo bài (`StratifiedKFold`) với đúng mô hình đó và so sánh.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, confusion_matrix

SHORT = {"vui_nang_luong_cao": "vui", "cang_toi_nang_luong_cao": "căng-tối",
         "buon_nang_luong_thap": "buồn", "binh_yen_nang_luong_thap": "bình yên"}
LABELS = ["vui", "căng-tối", "buồn", "bình yên"]

data["mood"] = data["quadrant"].map(SHORT)
X = data[feat_cols].to_numpy()
y = data["mood"].to_numpy()
groups = data["artist_id"].to_numpy()
genre = data["genre_coarse"].fillna("unknown").to_numpy()      # chỉ để làm MỐC so sánh, không đưa vào mô hình âm thanh

N_FOLDS, N_REPEATS = 4, 5
# mỗi lần lặp r dùng một cách xáo khác nhau (random_state = SEED + r)
folds_grouped = [list(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + r).split(X, y, groups))
                 for r in range(N_REPEATS)]
folds_random = [list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + r).split(X, y))
                for r in range(N_REPEATS)]

rows = []
for name, folds in [("theo artist", folds_grouped[0]), ("ngẫu nhiên theo bài", folds_random[0])]:   # chỉ in lần lặp đầu
    for k, (tr, te) in enumerate(folds):
        rows.append({"cách chia": name, "fold": k, "train": len(tr), "test": len(te),
                     "artist xuất hiện ở cả train và test": len(set(groups[tr]) & set(groups[te])),
                     **{f"test {l}": int((y[te] == l).sum()) for l in LABELS}})
display(pd.DataFrame(rows))

**Cách đọc bảng:** cột *artist xuất hiện ở cả train và test* phải bằng **0** với cách chia theo artist, và **lớn hơn 0** với cách chia ngẫu nhiên. Đó chính là rò rỉ.

### 2.2. Mô hình baseline và những quyết định nhỏ nhưng quan trọng

Baseline đầu tiên: **Logistic Regression** trên 93 đặc trưng đã chuẩn hóa. Chọn nó vì đơn giản và **giải thích được** (mỗi đặc trưng có một hệ số), phù hợp khi phải trả lời "mô hình dựa vào cái gì".

Ba quyết định:
1. **Chuẩn hóa nằm trong pipeline**: `StandardScaler` được *fit* chỉ trên phần train của mỗi fold, rồi áp lên test. Nếu fit trên cả tập thì trung bình/độ lệch của test đã lọt vào quá trình huấn luyện, một dạng rò rỉ khác.
2. **`class_weight="balanced"`**: nhóm "vui" nhiều gấp ~3 lần nhóm "căng-tối". Trọng số cân bằng khiến mô hình không bỏ qua nhóm nhỏ mà ta **không phải vứt bớt dữ liệu**.
3. **Chỉ số chính là macro-F1 và macro-recall** (trung bình đều giữa 4 nhóm), vì accuracy bị nhóm đông kéo.

Thêm **Random Forest** để so xem mô hình phi tuyến có hơn không. Nó không cần chuẩn hóa.

In [ ]:
def make_lr():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000))


def make_rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1)


class GenreOnly:
    # Mốc: chỉ biết thể loại lớn, đoán mood hay gặp nhất của thể loại đó (học trên train). Thể loại lạ -> nhóm đông nhất.
    def fit(self, g, y):
        s = pd.Series(y)
        self.table = s.groupby(np.asarray(g)).agg(lambda v: v.value_counts().idxmax()).to_dict()
        self.default = s.value_counts().idxmax()
        return self

    def predict(self, g):
        return np.array([self.table.get(x, self.default) for x in g])


def cv_eval(make_model, folds_by_repeat, X_, y_=y):
    # Với mỗi lần lặp và mỗi fold: fit trên train, dự đoán test.
    # Trả về (bảng điểm từng fold, mảng dự đoán out-of-fold kích thước [số lần lặp, số bài]).
    rows, oof = [], np.empty((len(folds_by_repeat), len(y_)), dtype=object)
    for r, folds in enumerate(folds_by_repeat):
        for k, (tr, te) in enumerate(folds):
            m = make_model().fit(X_[tr], y_[tr])
            p = m.predict(X_[te])
            oof[r, te] = p
            rows.append({"repeat": r, "fold": k,
                         "macro_f1": f1_score(y_[te], p, average="macro", labels=LABELS, zero_division=0),
                         "macro_recall": balanced_accuracy_score(y_[te], p),
                         "accuracy": accuracy_score(y_[te], p)})
    return pd.DataFrame(rows), oof


results = {}
results["đoán mò (theo tỉ lệ nhóm)"] = cv_eval(lambda: DummyClassifier(strategy="stratified", random_state=SEED), folds_grouped, X)
results["luôn đoán nhóm đông nhất"] = cv_eval(lambda: DummyClassifier(strategy="most_frequent"), folds_grouped, X)
results["chỉ dùng thể loại"] = cv_eval(GenreOnly, folds_grouped, genre)
results["âm thanh · LogReg · chia theo artist"] = cv_eval(make_lr, folds_grouped, X)
results["âm thanh · RandomForest · chia theo artist"] = cv_eval(make_rf, folds_grouped, X)
results["âm thanh · LogReg · chia NGẪU NHIÊN (rò rỉ)"] = cv_eval(make_lr, folds_random, X)


def summary_table(res):
    out = {}
    for name, (df_, _) in res.items():
        out[name] = {c: f"{df_[c].mean():.3f} ± {df_[c].std():.3f}" for c in ["macro_f1", "macro_recall", "accuracy"]}
    return pd.DataFrame(out).T


display(summary_table(results))

### 2.3. Cách đọc bảng kết quả

- **Mốc đoán mò và "luôn đoán nhóm đông nhất"**: với 4 nhóm, macro-recall của cả hai xoay quanh 0.25. Đây là sàn: mô hình không vượt sàn này thì chưa học được gì.
- **Mốc "chỉ dùng thể loại"**: đây là **mốc quan trọng nhất**. Mô hình âm thanh chỉ chứng tỏ giá trị riêng khi **vượt mốc này**; nếu không thì nó có thể chỉ đang nhận ra thể loại.
- **LogReg theo artist so với LogReg ngẫu nhiên**: chênh lệch giữa hai dòng này chính là **mức điểm ảo** do rò rỉ artist.
- Con số `± ...` là độ lệch chuẩn giữa 20 fold. Nếu hai mô hình hơn kém nhau **chỉ cỡ một độ lệch chuẩn** thì **chưa đủ cơ sở kết luận** cái nào tốt hơn. Phần 3 sẽ so sánh *ghép cặp* trên cùng các fold để chặt chẽ hơn.

### 2.4. Confusion matrix: mô hình nhầm nhóm nào với nhóm nào

Dùng dự đoán out-of-fold (mỗi bài được dự đoán khi nó nằm ở tập test), nên không dự đoán nào bị rò rỉ. Ta gộp cả 5 lần lặp cho ổn định. Mỗi hàng chia cho tổng hàng, nên ô chéo chính là **recall** của nhóm đó.

In [ ]:
oof_lr = results["âm thanh · LogReg · chia theo artist"][1]            # [N_REPEATS, số bài]
cm_n = sum(confusion_matrix(y, oof_lr[r], labels=LABELS) for r in range(N_REPEATS))   # cộng qua các lần lặp
cm = cm_n / cm_n.sum(axis=1, keepdims=True)                              # chia theo hàng -> recall

fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(LABELS, rotation=25)
ax.set_yticks(range(4)); ax.set_yticklabels(LABELS)
ax.set_xlabel("mô hình đoán"); ax.set_ylabel("nhãn thật")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm[i, j] > 0.5 else "black", fontsize=9)
ax.set_title("LogReg, chia theo artist\n(recall mỗi hàng, gộp 5 lần lặp)")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

print("Recall từng nhóm:", {l: round(float(cm[i, i]), 3) for i, l in enumerate(LABELS)})
print("\nConfusion matrix dạng bảng (hàng = nhãn thật, cột = mô hình đoán, mỗi hàng cộng = 1):")
display(pd.DataFrame(cm, index=LABELS, columns=LABELS).round(2))

**Hai trục của mood.** Bốn nhóm mood là tổ hợp của hai trục: **năng lượng** (*arousal*: cao hay thấp) và **cảm xúc** (*valence*: dương hay âm). "vui" = năng lượng cao + dương, "căng-tối" = cao + âm, "buồn" = thấp + âm, "bình yên" = thấp + dương. Từ confusion matrix ta gộp lại để xem mô hình nắm trục nào tốt hơn: với mỗi nhóm thật, cộng xác suất mô hình đoán vào **bất kỳ nhóm nào cùng phía** của trục đó (đoán mò = 0.5 với một trục hai phía).

In [ ]:
HIGH_AROUSAL = {"vui", "căng-tối"}       # năng lượng cao
POS_VALENCE = {"vui", "bình yên"}        # cảm xúc dương


def axis_recall(side_of):
    # Trung bình (qua 4 nhóm thật) xác suất mô hình đoán vào đúng phía của trục.
    hits = []
    for i, label in enumerate(LABELS):
        same_side = [j for j, other in enumerate(LABELS) if side_of(other) == side_of(label)]
        hits.append(cm[i, same_side].sum())
    return float(np.mean(hits))


print(f"Trục năng lượng (cao/thấp): {axis_recall(lambda m: m in HIGH_AROUSAL):.3f}   (đoán mò = 0.5)")
print(f"Trục cảm xúc (dương/âm):    {axis_recall(lambda m: m in POS_VALENCE):.3f}   (đoán mò = 0.5)")

---
## Phần 3 — Mô hình học mood hay học thể loại?

Ở notebook 01 ta thấy mood và thể loại **gắn chặt** (vd nhóm "bình yên" phần lớn là ambient). Nên có hai khả năng khi mô hình âm thanh đạt điểm cao hơn đoán mò:
- (a) nó thật sự nhận ra đặc điểm của **mood**;
- (b) nó chỉ nhận ra **thể loại** rồi suy ra mood.

Ta không phân biệt hoàn toàn được, nhưng có hai phép đo cho ta bằng chứng.

> **Đính chính con số ở notebook 01.** Mốc "chỉ dùng thể loại" ở đó (macro-recall khoảng 0.44) được tính **trên chính dữ liệu đó, không chia train/test**, và chỉ trên các bài thuộc đúng 1 thể loại lớn. Ở notebook này nó được tính **đúng cách** (học trên train, đo trên test), trên tập bài thật sự có audio, và bài **không rõ thể loại** (không thuộc đúng 1 thể loại lớn) được gom vào một nhóm `unknown`. Hai con số vì vậy **không so với nhau được**; hãy dùng con số của notebook này.

### 3.1. Chia theo "thể loại có rõ hay không"
Với bài **thể loại rõ**, mốc "chỉ dùng thể loại" có thông tin để đoán. Với bài `unknown`, mốc đó chỉ còn cách đoán nhóm đông nhất, còn mô hình âm thanh vẫn nghe được. Nếu mô hình âm thanh chỉ hơn mốc ở bài `unknown` mà **không hơn** ở bài thể loại rõ, thì phần lớn lợi thế của nó đến từ chỗ mốc kia "mù", chứ không phải nó biết thêm gì ngoài thể loại.

In [ ]:
known = genre != "unknown"
print(f"Bài thể loại rõ: {int(known.sum())} | bài không rõ (unknown): {int((~known).sum())}")


def subset_scores(oof_by_repeat, mask):
    # Tính điểm riêng cho từng lần lặp trên tập con `mask`, rồi lấy trung bình các lần lặp.
    rows = [{"macro_recall": balanced_accuracy_score(y[mask], o[mask]),
             "macro_f1": f1_score(y[mask], o[mask], average="macro", labels=LABELS, zero_division=0),
             "accuracy": accuracy_score(y[mask], o[mask])} for o in oof_by_repeat]
    return pd.DataFrame(rows).mean()


compare = {
    "chỉ dùng thể loại": results["chỉ dùng thể loại"][1],
    "âm thanh · LogReg": results["âm thanh · LogReg · chia theo artist"][1],
    "âm thanh · RandomForest": results["âm thanh · RandomForest · chia theo artist"][1],
}
tab = {}
for name, oof_ in compare.items():
    for sub_name, mask in [(f"thể loại rõ (n={int(known.sum())})", known), (f"unknown (n={int((~known).sum())})", ~known)]:
        s_ = subset_scores(oof_, mask)
        tab[(sub_name, name)] = {"macro_recall": round(s_["macro_recall"], 3),
                                 "macro_f1": round(s_["macro_f1"], 3), "accuracy": round(s_["accuracy"], 3)}
display(pd.DataFrame(tab).T)

**Cách đọc:** so sánh các dòng *cùng một tập con*. Chú ý cột `macro_recall` (đoán mò = 0.25).

### 3.2. So sánh ghép cặp trên cùng các fold
Điểm trung bình có thể đánh lừa vì mỗi fold có độ khó khác nhau. So sánh **ghép cặp** (cùng fold, hiệu của hai mô hình) cho bằng chứng chặt chẽ hơn. Ta xem hiệu *âm thanh − chỉ thể loại*, và **tỉ lệ fold mà mô hình âm thanh thắng**.

In [ ]:
def paired(a_name, b_name, metric="macro_recall"):
    a = results[a_name][0].set_index(["repeat", "fold"])[metric]
    b = results[b_name][0].set_index(["repeat", "fold"])[metric]
    d = a - b
    return {"so sánh": f"{a_name}  −  {b_name}", "hiệu trung bình": round(d.mean(), 3),
            "độ lệch chuẩn hiệu": round(d.std(), 3), "% fold mà vế trái thắng": f"{(d > 0).mean():.0%}"}


GEN, LR_, RF_ = "chỉ dùng thể loại", "âm thanh · LogReg · chia theo artist", "âm thanh · RandomForest · chia theo artist"
CH = "đoán mò (theo tỉ lệ nhóm)"
display(pd.DataFrame([paired(LR_, CH), paired(LR_, GEN), paired(RF_, GEN), paired(RF_, LR_)]).set_index("so sánh"))

**Cách đọc:** "% fold mà vế trái thắng" gần 100% và hiệu lớn hơn nhiều độ lệch chuẩn thì khá chắc; quanh 50% hoặc hiệu nhỏ hơn độ lệch chuẩn thì **chưa phân biệt được**. Lưu ý 20 fold này chung dữ liệu nên **không độc lập hoàn toàn**; các con số cho thấy xu hướng, không phải kiểm định thống kê chặt.

---
## Phần 4 — Ablation: nhóm đặc trưng nào đáng giá?

Ta chạy lại LogReg (cùng cách chia theo artist) với **từng nhóm đặc trưng riêng** và với **bỏ từng nhóm**:
- **Chỉ nhóm X:** nhóm X tự nó có sức dự đoán không?
- **Bỏ nhóm X:** nhóm X có **đóng góp riêng** không? Nếu bỏ đi mà điểm gần như không đổi, các nhóm khác đã chứa thông tin tương tự.

Hai phép đo trả lời hai câu hỏi khác nhau; đọc cả hai.

In [ ]:
GROUPS_ = ["mfcc", "chroma", "spectral", "energy", "rhythm"]
col_group = pd.Series({c: feature_group(c) for c in feat_cols})


def ablate(cols):
    idx = [feat_cols.index(c) for c in cols]
    return cv_eval(make_lr, folds_grouped, X[:, idx])[0]


abl = {"tất cả": (len(feat_cols), ablate(feat_cols))}
for g_ in GROUPS_:
    cols = list(col_group[col_group == g_].index)
    abl[f"chỉ {g_}"] = (len(cols), ablate(cols))
for g_ in GROUPS_:
    cols = list(col_group[col_group != g_].index)
    abl[f"bỏ {g_}"] = (len(cols), ablate(cols))

abl_tab = pd.DataFrame({k: {"số chiều": n, "macro_f1": f"{d_['macro_f1'].mean():.3f} ± {d_['macro_f1'].std():.3f}",
                            "macro_recall": f"{d_['macro_recall'].mean():.3f} ± {d_['macro_recall'].std():.3f}"}
                        for k, (n, d_) in abl.items()}).T
display(abl_tab)

fig, ax = plt.subplots(figsize=(7, 4.2))
names = list(abl.keys())[::-1]
means = [abl[k][1]["macro_f1"].mean() for k in names]
stds = [abl[k][1]["macro_f1"].std() for k in names]
ax.barh(names, means, xerr=stds, color=["#4C78A8" if n.startswith("chỉ") else "#F58518" if n.startswith("bỏ") else "#54A24B" for n in names],
        capsize=3)
ax.axvline(results[CH][0]["macro_f1"].mean(), color="gray", ls="--", label="đoán mò")
ax.axvline(results[GEN][0]["macro_f1"].mean(), color="black", ls=":", label="chỉ dùng thể loại")
ax.set_xlabel("macro-F1 (trung bình ± độ lệch chuẩn giữa 20 fold)"); ax.legend(loc="lower right", fontsize=8)
ax.set_title("Ablation nhóm đặc trưng (LogReg, chia theo artist)")
plt.tight_layout(); plt.show()

---
## Phần 5 — Mô hình dựa vào đặc trưng nào?

Vì dùng Logistic Regression trên đặc trưng đã chuẩn hóa, mỗi nhóm mood có một vector hệ số: hệ số **dương lớn** nghĩa là đặc trưng đó càng cao càng làm mô hình nghiêng về nhóm ấy. Ô dưới huấn luyện lại trên **toàn bộ** dữ liệu *chỉ để xem hệ số* (không dùng để đo điểm) và in 5 đặc trưng có hệ số dương lớn nhất cho từng nhóm.

> Cẩn trọng khi đọc: nhiều đặc trưng **tương quan với nhau** (vd các MFCC, trung bình và độ lệch chuẩn của cùng một đặc trưng), nên trọng số bị chia sẻ giữa chúng. Hãy dùng hệ số để **gợi ý giả thuyết**, đừng coi là bằng chứng nhân quả.

In [ ]:
final = make_lr().fit(X, y)
clf = final[-1]
coef = pd.DataFrame(clf.coef_.T, index=feat_cols, columns=clf.classes_)
for cls in LABELS:
    top = coef[cls].sort_values(ascending=False).head(5)
    print(f"{cls:9s}: " + ", ".join(f"{n} ({v:+.2f})" for n, v in top.items()))

---
## Hạn chế cần nói thật khi vấn đáp

- **Dữ liệu nhỏ:** khoảng một nghìn bài, 299 artist, nhóm nhỏ nhất 127 bài. Độ lệch chuẩn giữa các fold lớn, nên chênh lệch nhỏ giữa hai mô hình **không đủ để kết luận**.
- **Nhãn nhiễu:** mood là **tag do người dùng gán**, không phải đo đạc. Ta còn bỏ các bài có tag thuộc nhiều nhóm, nên tập đang xét là các bài "rõ ràng" hơn thực tế.
- **Cách gom tag là quyết định thiết kế** (`MOOD_QUADRANTS`), chưa kiểm chứng với người nghe; đổi cách gom thì kết quả đổi.
- **Chỉ 30 giây giữa bài**, giả định mood không đổi trong bài.
- **Thể loại `unknown` chiếm phần đáng kể**, nên mốc "chỉ dùng thể loại" phụ thuộc vào cách gom thể loại.
- **Chưa tinh chỉnh siêu tham số** (`C` của LogReg, cấu hình RandomForest). Nếu tinh chỉnh thì phải dùng *nested cross-validation*, nếu không điểm bị lạc quan.

**Hướng tiếp theo:** tải thêm audio (nhất là nhóm "căng-tối"), thử cách gom mood khác, thử embedding pretrained như **một mốc so sánh** (không phải lõi), và quay lại hướng B (fingerprint) nếu còn thời gian.